# Hugging Face Transformers 綜合實戰指南

> 從入門到精通：涵蓋文本分類、NER、摘要、問答等多個應用

## 📚 本教程內容

1. **環境設置與基礎概念**
2. **Pipeline API 快速上手**
3. **文本分類 (情感分析、主題分類)**
4. **命名實體識別 (NER)**
5. **文本摘要 (抽取式與生成式)**
6. **問答系統 (QA)**
7. **模型微調實戰**
8. **模型保存與部署**
9. **性能優化技巧**

## 🎯 學習目標

- 掌握 Hugging Face Transformers 庫的使用
- 了解如何快速應用預訓練模型
- 學會微調模型適配自己的任務
- 理解不同 NLP 任務的最佳實踐

## 1. 環境設置

### 安裝必要的庫

In [ ]:
# 安裝 Hugging Face 生態系統
!pip install -q transformers datasets evaluate accelerate sentencepiece

# 可視化工具
!pip install -q matplotlib seaborn pandas

In [ ]:
# 導入必要的庫
import torch
import numpy as np
import pandas as pd
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoModelForQuestionAnswering,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset, Dataset
import evaluate

# 設置隨機種子
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# 檢查 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 使用設備: {device}")
print(f"📦 Transformers 版本: {__import__('transformers').__version__}")

## 2. Pipeline API 快速上手

Pipeline 是最簡單的使用方式，適合快速驗證和原型開發。

### 2.1 情感分析

In [ ]:
# 創建情感分析 pipeline
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# 測試
texts = [
    "I love this product! It's amazing!",
    "This is the worst experience ever.",
    "The movie was okay, nothing special."
]

results = sentiment_analyzer(texts)

for text, result in zip(texts, results):
    print(f"文本: {text}")
    print(f"情感: {result['label']} (置信度: {result['score']:.4f})\n")

### 2.2 零樣本分類

不需要訓練，直接指定類別標籤進行分類！

In [ ]:
# 零樣本分類 - 非常強大！
zero_shot_classifier = pipeline("zero-shot-classification")

text = "I'm looking for a new laptop for machine learning."
candidate_labels = ["technology", "sports", "politics", "entertainment"]

result = zero_shot_classifier(text, candidate_labels)

print(f"文本: {text}\n")
for label, score in zip(result['labels'], result['scores']):
    print(f"{label}: {score:.4f}")

### 2.3 文本生成

In [ ]:
# 文本生成
generator = pipeline("text-generation", model="gpt2")

prompt = "Natural language processing is"
outputs = generator(
    prompt,
    max_length=50,
    num_return_sequences=2,
    temperature=0.8
)

print(f"Prompt: {prompt}\n")
for i, output in enumerate(outputs, 1):
    print(f"生成 {i}: {output['generated_text']}\n")

## 3. 文本分類實戰

### 3.1 使用預訓練模型

In [ ]:
# 載入模型和分詞器
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
).to(device)

# 測試分詞
text = "This is a great product!"
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

print("分詞結果:")
print(f"Input IDs: {inputs['input_ids']}")
print(f"Attention Mask: {inputs['attention_mask']}")
print(f"\n解碼: {tokenizer.decode(inputs['input_ids'][0])}")

### 3.2 模型微調 - IMDb 情感分析

In [ ]:
# 載入數據集（使用小樣本以加快速度）
dataset = load_dataset("imdb", split="train[:1000]")
dataset = dataset.train_test_split(test_size=0.2)

print(f"訓練集大小: {len(dataset['train'])}")
print(f"測試集大小: {len(dataset['test'])}")
print(f"\n樣本示例:")
print(dataset['train'][0])

In [ ]:
# 數據預處理
def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 檢查處理後的數據
print("處理後的特徵:")
print(tokenized_datasets['train'].column_names)

In [ ]:
# 定義評估指標
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# 訓練配置
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir='./logs',
    logging_steps=50,
)

# 創建 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# 開始訓練
print("🚀 開始訓練...")
trainer.train()

In [ ]:
# 評估模型
eval_results = trainer.evaluate()
print(f"\n📊 評估結果:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

In [ ]:
# 使用微調後的模型進行預測
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    prediction = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][prediction].item()
    
    label = "Positive" if prediction == 1 else "Negative"
    return label, confidence

# 測試
test_texts = [
    "This movie is absolutely fantastic!",
    "Worst film I've ever seen.",
    "It was okay, not great but not terrible."
]

for text in test_texts:
    label, confidence = predict_sentiment(text)
    print(f"文本: {text}")
    print(f"預測: {label} (置信度: {confidence:.4f})\n")

## 4. 命名實體識別 (NER)

識別文本中的實體，如人名、地名、組織等。

### 4.1 使用預訓練 NER 模型

In [ ]:
# 創建 NER pipeline
ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"  # 合併相鄰的同類實體
)

# 測試
text = "Apple Inc. CEO Tim Cook announced a new product in San Francisco."
entities = ner(text)

print(f"文本: {text}\n")
print("識別的實體:")
for entity in entities:
    print(f"  {entity['word']:20s} | {entity['entity_group']:10s} | 分數: {entity['score']:.4f}")

### 4.2 可視化 NER 結果

In [ ]:
def visualize_ner(text, entities):
    """可視化 NER 結果"""
    # 創建彩色標註
    colors = {
        'PER': '\033[91m',  # 紅色 - 人名
        'ORG': '\033[92m',  # 綠色 - 組織
        'LOC': '\033[93m',  # 黃色 - 地點
        'MISC': '\033[94m', # 藍色 - 其他
    }
    reset = '\033[0m'
    
    result = text
    offset = 0
    
    for entity in sorted(entities, key=lambda x: x['start']):
        start = entity['start'] + offset
        end = entity['end'] + offset
        entity_type = entity['entity_group']
        
        color = colors.get(entity_type, reset)
        replacement = f"{color}{entity['word']}({entity_type}){reset}"
        
        result = result[:start] + replacement + result[end:]
        offset += len(replacement) - (end - start)
    
    print(result)

visualize_ner(text, entities)

## 5. 文本摘要

### 5.1 抽取式摘要 vs 生成式摘要

- **抽取式**: 從原文中選取關鍵句子
- **生成式**: 生成新的摘要文本

In [ ]:
# 使用 BART 進行生成式摘要
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# 長文本示例
article = """
Natural language processing (NLP) is a subfield of linguistics, computer science, 
and artificial intelligence concerned with the interactions between computers and 
human language. NLP is used to apply machine learning algorithms to text and speech. 
For example, we can use NLP to create systems like speech recognition, document 
summarization, machine translation, spam detection, named entity recognition, 
question answering, autocomplete, predictive typing and so on. In recent years, 
deep learning approaches have achieved state-of-the-art results in many NLP tasks.
"""

# 生成摘要
summary = summarizer(
    article,
    max_length=50,
    min_length=25,
    do_sample=False
)

print("原文:")
print(article)
print("\n摘要:")
print(summary[0]['summary_text'])

## 6. 問答系統 (Question Answering)

從給定的上下文中抽取答案。

In [ ]:
# 創建 QA pipeline
qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

# 上下文和問題
context = """
The Transformer architecture was introduced in the paper 'Attention Is All You Need' 
by Vaswani et al. in 2017. It relies entirely on self-attention mechanisms to compute 
representations of its input and output without using sequence-aligned RNNs or convolution. 
The Transformer has been widely adopted in NLP and has led to models like BERT and GPT.
"""

questions = [
    "When was the Transformer introduced?",
    "What mechanism does Transformer rely on?",
    "What are some models based on Transformer?"
]

print(f"上下文: {context}\n")
for question in questions:
    result = qa(question=question, context=context)
    print(f"問題: {question}")
    print(f"答案: {result['answer']} (分數: {result['score']:.4f})\n")

## 7. 多任務模型

一些模型可以處理多種任務，如 T5。

In [ ]:
# T5 是一個統一的 text-to-text 模型
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

# T5 使用前綴來指定任務
tasks = [
    "translate English to German: Hello, how are you?",
    "summarize: Natural language processing is a field of AI that focuses on the interaction between computers and humans through natural language.",
    "question: What is NLP? context: Natural language processing (NLP) is a subfield of AI."
]

for task in tasks:
    inputs = tokenizer(task, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_length=50)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"任務: {task}")
    print(f"結果: {result}\n")

## 8. 模型保存與加載

### 8.1 保存微調後的模型

In [ ]:
# 保存模型
save_directory = "./my_finetuned_model"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"✅ 模型已保存到 {save_directory}")

In [ ]:
# 加載模型
loaded_model = AutoModelForSequenceClassification.from_pretrained(save_directory)
loaded_tokenizer = AutoTokenizer.from_pretrained(save_directory)

print("✅ 模型加載成功")

### 8.2 上傳到 Hugging Face Hub (可選)

In [ ]:
# 登錄 Hugging Face
# from huggingface_hub import login
# login()

# 上傳模型
# model.push_to_hub("my-awesome-model")
# tokenizer.push_to_hub("my-awesome-model")

print("💡 提示: 需要 Hugging Face 賬號和 token")

## 9. 性能優化技巧

### 9.1 使用 DistilBERT (更快、更小)

In [ ]:
import time

# 比較 BERT 和 DistilBERT
models_to_compare = [
    "bert-base-uncased",
    "distilbert-base-uncased"
]

text = "This is a test sentence for performance comparison."

for model_name in models_to_compare:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2
    ).to(device)
    
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    # 測量推理時間
    start = time.time()
    with torch.no_grad():
        for _ in range(100):
            outputs = model(**inputs)
    end = time.time()
    
    avg_time = (end - start) / 100 * 1000  # 轉換為毫秒
    num_params = sum(p.numel() for p in model.parameters())
    
    print(f"\n{model_name}:")
    print(f"  參數量: {num_params:,}")
    print(f"  平均推理時間: {avg_time:.2f} ms")

### 9.2 批量處理

In [ ]:
# 批量處理比單條處理快很多
texts = [f"Sample text {i}" for i in range(32)]

# 單條處理
start = time.time()
for text in texts:
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
single_time = time.time() - start

# 批量處理
start = time.time()
inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
with torch.no_grad():
    outputs = model(**inputs)
batch_time = time.time() - start

print(f"單條處理時間: {single_time:.4f}s")
print(f"批量處理時間: {batch_time:.4f}s")
print(f"速度提升: {single_time/batch_time:.2f}x")

### 9.3 模型量化 (可選)

In [ ]:
# 動態量化可以減小模型大小和加快推理
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("✅ 模型已量化")
print(f"原始模型大小: {sum(p.numel() * p.element_size() for p in model.parameters()) / 1024 / 1024:.2f} MB")
print(f"量化模型大小: {sum(p.numel() * p.element_size() for p in quantized_model.parameters()) / 1024 / 1024:.2f} MB")

## 10. 實用技巧總結

### 選擇合適的模型

| 任務 | 推薦模型 | 特點 |
|------|---------|------|
| 文本分類 | BERT, RoBERTa, DistilBERT | 平衡性能和速度 |
| NER | BERT, RoBERTa | 理解上下文 |
| 摘要 | BART, T5, Pegasus | 生成式任務 |
| 問答 | BERT, RoBERTa, DeBERTa | 精確定位 |
| 文本生成 | GPT-2, GPT-3, T5 | 流暢生成 |
| 翻譯 | mBART, T5, MarianMT | 多語言支持 |

### 最佳實踐

1. **從預訓練模型開始**: 幾乎總是比從頭訓練好
2. **選擇合適的模型大小**: DistilBERT 通常是個好起點
3. **數據質量 > 數據量**: 清洗數據很重要
4. **使用學習率調度器**: 如 linear warmup
5. **早停**: 避免過擬合
6. **批量處理**: 提高效率
7. **監控指標**: 不只是 loss，還要看 F1, accuracy 等
8. **嘗試不同的超參數**: 學習率、batch size、epoch 數

### 常見問題

**Q: CUDA out of memory?**
- 減小 batch size
- 使用梯度累積
- 使用更小的模型
- 使用混合精度訓練

**Q: 訓練太慢?**
- 使用 GPU
- 減小序列長度
- 使用 DistilBERT
- 使用數據並行

**Q: 模型過擬合?**
- 增加 dropout
- 使用數據增強
- 減少訓練輪數
- 收集更多數據

## 11. 下一步學習

### 進階主題

1. **多任務學習**: 一個模型處理多個任務
2. **零樣本/少樣本學習**: Prompt Engineering
3. **多語言模型**: mBERT, XLM-RoBERTa
4. **大型語言模型**: GPT-3, GPT-4, Claude
5. **RAG (檢索增強生成)**: 結合檢索和生成

### 參考資源

- [Hugging Face Course](https://huggingface.co/course)
- [Transformers Documentation](https://huggingface.co/docs/transformers)
- [Papers with Code](https://paperswithcode.com/)
- [NLP Progress](http://nlpprogress.com/)

---

**🎉 恭喜你完成了 Hugging Face Transformers 綜合教程！**

現在你已經掌握了：
- ✅ Pipeline API 的使用
- ✅ 多種 NLP 任務的實現
- ✅ 模型微調的完整流程
- ✅ 性能優化技巧

繼續探索和實踐吧！🚀